## Cleaning and preparing the Movie file (movies_raw)
Movies: Extracted Year and Title from the text and converted Genres into a List.

    Example: "Toy Story (1995)" → pure_title: "Toy Story", release_year: 1995

    Example: "Action|Sci-Fi" → ["Action", "Sci-Fi"]

Ratings: Converted Unix Seconds into a standard Date.

    Example: 964981208 → 2000-07-30

Transformation: Split the genres string into an Array using the pipe (|) delimiter.

Example: "Adventure|Comedy" → ["Adventure", "Comedy"]

Benefit: Allows for easy filtering (e.g., "Find all movies where 'Comedy' is in the list").


In [0]:
from pyspark.sql.functions import col, split, regexp_extract, trim

# 1. Pull the data from the Bronze table we created earlier
df_bronze_movies = spark.table("workspace.movielens_bronze.movies_raw")

# 2. Transform: Cleaning Titles and Genres
df_movies_clean = (df_bronze_movies
    # Extract the year: Looking for 4 digits inside ()
    .withColumn("release_year", regexp_extract(col("title"), r"\((\d{4})\)", 1))
    
    # Extract the title: Everything before the year
    .withColumn("pure_title", trim(regexp_extract(col("title"), r"(.*)\s\(", 1)))
    
    # Transform Genres: Turn "Action|Sci-Fi" into ["Action", "Sci-Fi"]
    .withColumn("genres_array", split(col("genres"), "\|"))
)

# 3. Check the "Surgery" results
display(df_movies_clean.select("title", "pure_title", "release_year", "genres_array").limit(10))

<>:15: SyntaxWarning: invalid escape sequence '\|'
<>:15: SyntaxWarning: invalid escape sequence '\|'
/home/spark-3ea680c5-62da-46da-80be-1a/.ipykernel/2564/command-5586297476154984-2289395653:15: SyntaxWarning: invalid escape sequence '\|'
  .withColumn("genres_array", split(col("genres"), "\|"))
<unknown>:15: SyntaxWarning: invalid escape sequence '\|'


title,pure_title,release_year,genres_array
Toy Story (1995),Toy Story,1995,"List(Adventure, Animation, Children, Comedy, Fantasy)"
Jumanji (1995),Jumanji,1995,"List(Adventure, Children, Fantasy)"
Grumpier Old Men (1995),Grumpier Old Men,1995,"List(Comedy, Romance)"
Waiting to Exhale (1995),Waiting to Exhale,1995,"List(Comedy, Drama, Romance)"
Father of the Bride Part II (1995),Father of the Bride Part II,1995,List(Comedy)
Heat (1995),Heat,1995,"List(Action, Crime, Thriller)"
Sabrina (1995),Sabrina,1995,"List(Comedy, Romance)"
Tom and Huck (1995),Tom and Huck,1995,"List(Adventure, Children)"
Sudden Death (1995),Sudden Death,1995,List(Action)
GoldenEye (1995),GoldenEye,1995,"List(Action, Adventure, Thriller)"


<unknown>:15: SyntaxWarning: invalid escape sequence '\|'


## ## Cleaning and preparing the Rating file (rating_raw)


In [0]:
from pyspark.sql.functions import from_unixtime, col

# 1. Load the raw ratings
df_bronze_ratings = spark.table("workspace.movielens_bronze.ratings_raw")

# 2. Transform: Seconds to Date
df_ratings_clean = df_bronze_ratings.withColumn(
    "rating_date", 
    from_unixtime(col("timestamp")).cast("date")
)

# 3. Check the result
display(df_ratings_clean.select("userId", "movieId", "rating", "timestamp", "rating_date").limit(10))

userId,movieId,rating,timestamp,rating_date
1,1,4.0,964982703,2000-07-30
1,3,4.0,964981247,2000-07-30
1,6,4.0,964982224,2000-07-30
1,47,5.0,964983815,2000-07-30
1,50,5.0,964982931,2000-07-30
1,70,3.0,964982400,2000-07-30
1,101,5.0,964980868,2000-07-30
1,110,4.0,964982176,2000-07-30
1,151,5.0,964984041,2000-07-30
1,157,5.0,964984100,2000-07-30


##preparing the Links Table

In [0]:
# Load the raw links from Bronze
df_links = spark.table("workspace.movielens_bronze.links_raw")

# We don't need to change much here, but let's see what we have
display(df_links.limit(5))

movieId,imdbId,tmdbId
1,114709,862
2,113497,8844
3,113228,15602
4,114885,31357
5,113041,11862


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.movielens_silver;

#The Grand Join (Merging it all)


In [0]:
# 1. Start with the Movies table as the "Base"
# 2. Left Join Ratings: This keeps movies with NO ratings (fulfills "Data Quality" requirement)
# 3. Left Join Links: This keeps movies even if they have no IMDb/TMDb link
df_silver_master = (df_movies_clean
    .join(df_ratings_clean, on="movieId", how="left")
    .join(df_links, on="movieId", how="left")
)

# 4. Save to Silver Schema
df_silver_master.write.mode("overwrite").saveAsTable("workspace.movielens_silver.fact_ratings")

print("✅ Silver Master Table updated with Left Joins and Movie-first logic!")
display(df_silver_master.limit(10))

✅ Silver Master Table updated with Left Joins and Movie-first logic!


movieId,title,genres,release_year,pure_title,genres_array,userId,rating,timestamp,rating_date,imdbId,tmdbId
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,Toy Story,"List(Adventure, Animation, Children, Comedy, Fantasy)",610,5.0,1479542900,2016-11-19,114709,862
2,Jumanji (1995),Adventure|Children|Fantasy,1995,Jumanji,"List(Adventure, Children, Fantasy)",608,2.0,1117490786,2005-05-30,113497,8844
3,Grumpier Old Men (1995),Comedy|Romance,1995,Grumpier Old Men,"List(Comedy, Romance)",608,2.0,1117504413,2005-05-31,113228,15602
4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995,Waiting to Exhale,"List(Comedy, Drama, Romance)",600,1.5,1237760055,2009-03-22,114885,31357
5,Father of the Bride Part II (1995),Comedy,1995,Father of the Bride Part II,List(Comedy),604,3.0,832080355,1996-05-14,113041,11862
6,Heat (1995),Action|Crime|Thriller,1995,Heat,"List(Action, Crime, Thriller)",610,5.0,1493850345,2017-05-03,113277,949
7,Sabrina (1995),Comedy|Romance,1995,Sabrina,"List(Comedy, Romance)",606,2.5,1171754710,2007-02-17,114319,11860
8,Tom and Huck (1995),Adventure|Children,1995,Tom and Huck,"List(Adventure, Children)",501,3.0,844974090,1996-10-10,112302,45325
9,Sudden Death (1995),Action,1995,Sudden Death,List(Action),599,1.5,1498504960,2017-06-26,114576,9091
10,GoldenEye (1995),Action|Adventure|Thriller,1995,GoldenEye,"List(Action, Adventure, Thriller)",609,4.0,847220937,1996-11-05,113189,710


In [0]:
# 1. Load the Master Table
df_master = spark.table("workspace.movielens_silver.fact_ratings")

# 2. Print just the column names as a list
print("Master Table Columns:", df_master.columns)

# 3. Show the full table preview
display(df_master)

Master Table Columns: ['movieId', 'userId', 'rating', 'timestamp', 'rating_date', 'title', 'genres', 'release_year', 'pure_title', 'genres_array', 'imdbId', 'tmdbId']


movieId,userId,rating,timestamp,rating_date,title,genres,release_year,pure_title,genres_array,imdbId,tmdbId
1,610,5.0,1479542900,2016-11-19,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,Toy Story,"List(Adventure, Animation, Children, Comedy, Fantasy)",114709,862
2,608,2.0,1117490786,2005-05-30,Jumanji (1995),Adventure|Children|Fantasy,1995,Jumanji,"List(Adventure, Children, Fantasy)",113497,8844
3,608,2.0,1117504413,2005-05-31,Grumpier Old Men (1995),Comedy|Romance,1995,Grumpier Old Men,"List(Comedy, Romance)",113228,15602
4,600,1.5,1237760055,2009-03-22,Waiting to Exhale (1995),Comedy|Drama|Romance,1995,Waiting to Exhale,"List(Comedy, Drama, Romance)",114885,31357
5,604,3.0,832080355,1996-05-14,Father of the Bride Part II (1995),Comedy,1995,Father of the Bride Part II,List(Comedy),113041,11862
6,610,5.0,1493850345,2017-05-03,Heat (1995),Action|Crime|Thriller,1995,Heat,"List(Action, Crime, Thriller)",113277,949
7,606,2.5,1171754710,2007-02-17,Sabrina (1995),Comedy|Romance,1995,Sabrina,"List(Comedy, Romance)",114319,11860
8,501,3.0,844974090,1996-10-10,Tom and Huck (1995),Adventure|Children,1995,Tom and Huck,"List(Adventure, Children)",112302,45325
9,599,1.5,1498504960,2017-06-26,Sudden Death (1995),Action,1995,Sudden Death,List(Action),114576,9091
10,609,4.0,847220937,1996-11-05,GoldenEye (1995),Action|Adventure|Thriller,1995,GoldenEye,"List(Action, Adventure, Thriller)",113189,710
